In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
import torch
import timm
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [3]:
class CFG:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    IMG_SIZE = 224
    BATCH_SIZE = 8
    NUM_FRAMES = 8
    MODEL_PATH = "/content/drive/MyDrive/deepfake_project/best_true_late_fusion.pth"
    DATA_ROOT = "/content/drive/MyDrive/deepfake_project/data/cropped_faces_celeb_df"
    META_CSV = "/content/drive/MyDrive/deepfake_project/data/metadata/frame_metadata_celeb_df.csv"

print("Device:", CFG.DEVICE)

Device: cuda


In [4]:
transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(),
    ToTensorV2()
])

In [5]:
class CelebDFVideoDataset(Dataset):
    """
    Groups cropped face frames into video-level samples
    """

    def __init__(self, root_dir, transform=None, num_frames=8):
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        self.video_dict = defaultdict(list)

        for label_name, label in [("real", 0), ("fake", 1)]:
            split_dir = os.path.join(root_dir, label_name)

            if not os.path.exists(split_dir):
                continue

            for video_id in os.listdir(split_dir):
                video_dir = os.path.join(split_dir, video_id)

                if not os.path.isdir(video_dir):
                    continue

                for frame in os.listdir(video_dir):
                    if frame.endswith(".jpg"):
                        self.video_dict[video_id].append({
                            "path": os.path.join(video_dir, frame),
                            "label": label
                        })

        self.video_ids = list(self.video_dict.keys())

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        vid = self.video_ids[idx]
        samples = self.video_dict[vid]
        label = samples[0]["label"]

        # sample frames
        if len(samples) >= self.num_frames:
            selected = np.random.choice(samples, self.num_frames, replace=False)
        else:
            selected = np.random.choice(samples, self.num_frames, replace=True)

        frames = []

        for s in selected:
            img = cv2.imread(s["path"])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            if self.transform:
                img = self.transform(image=img)["image"]

            frames.append(img)

        frames = torch.stack(frames)  # [T, C, H, W]

        return {
            "frames": frames,
            "label": torch.tensor(label, dtype=torch.float32),
            "video_id": vid
        }


In [6]:
class TrueLateFusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "tf_efficientnetv2_b0",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )

        feat_dim = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        feat = self.backbone(x)

        feat = feat.view(B, T, -1)

        # TRUE LATE FUSION
        fused = feat.mean(dim=1)

        logits = self.classifier(fused).squeeze(1)

        return logits

In [7]:
dataset = CelebDFVideoDataset(
    CFG.DATA_ROOT,
    transform=transform,
    num_frames=CFG.NUM_FRAMES
)

loader = DataLoader(
    dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Total videos:", len(dataset))

Total videos: 600


In [8]:
model = TrueLateFusionModel().to(CFG.DEVICE)

model.load_state_dict(
    torch.load(CFG.MODEL_PATH, map_location=CFG.DEVICE)
)

model.eval()

sigmoid = nn.Sigmoid()

In [9]:
all_probs = []
all_preds = []
all_labels = []
all_vids = []

with torch.no_grad():

    for batch in tqdm(loader):

        frames = batch["frames"].to(CFG.DEVICE)
        labels = batch["label"].cpu().numpy()
        vids = batch["video_id"]

        logits = model(frames)
        probs = sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels)
        all_vids.extend(vids)

100%|██████████| 75/75 [12:58<00:00, 10.38s/it]


In [10]:
print("\n=== CELEB-DF TRUE LATE FUSION RESULTS ===\n")

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("ROC-AUC  :", auc)


print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, digits=4))


cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n", cm)



=== CELEB-DF TRUE LATE FUSION RESULTS ===

Accuracy : 0.5766666666666667
Precision: 0.7804878048780488
Recall   : 0.21333333333333335
F1 Score : 0.33507853403141363
ROC-AUC  : 0.7095

Classification Report:

              precision    recall  f1-score   support

         0.0     0.5444    0.9400    0.6895       300
         1.0     0.7805    0.2133    0.3351       300

    accuracy                         0.5767       600
   macro avg     0.6624    0.5767    0.5123       600
weighted avg     0.6624    0.5767    0.5123       600


Confusion Matrix:
 [[282  18]
 [236  64]]


## Celeb-DF Inference with Late Fusion: Results and Analysis

The True Late Fusion model was evaluated on the Celeb-DF dataset using cropped face sequences extracted from real and fake videos. The model achieved an accuracy of 57.67%, which is slightly better than random guessing on this balanced dataset. Since the dataset contains an equal number of real and fake videos, a random classifier would achieve around 50% accuracy. This result shows that the model learned some meaningful deepfake-related patterns, but its ability to generalize to a new dataset is still limited.

The model achieved a precision score of 78.05%. Precision measures how often the model is correct when it predicts that a video is fake. This relatively high precision indicates that the model is careful when labeling videos as fake and usually makes correct fake predictions. However, the recall score was only 21.33%, which means the model failed to detect many fake videos. Out of 300 fake videos, the model correctly identified only 64 and incorrectly classified 236 fake videos as real. This shows that the model is strongly biased toward predicting videos as real.

The F1-score of 0.335 reflects the imbalance between precision and recall. Although the model makes reliable fake predictions when it is confident, it misses most fake samples, resulting in a low overall balance between detection accuracy and sensitivity. This indicates that the current model is not yet reliable enough for practical deepfake detection tasks.

One of the most important metrics in this evaluation is the ROC-AUC score, which was 0.7095. The ROC-AUC score measures the model’s ability to separate real and fake videos regardless of the chosen classification threshold. A value of 0.5 would indicate random predictions, while higher values indicate stronger discrimination ability. The achieved ROC-AUC score suggests that the model learned useful features and can partially distinguish fake videos from real ones. Therefore, the model is not behaving randomly, even though its final classification performance is weak.

The confusion matrix provides additional insight into the model’s behavior. Among the 300 real videos, the model correctly classified 282 videos and incorrectly labeled 18 videos as fake. This corresponds to a very high real-video detection rate of approximately 94%. However, for fake videos, the model correctly detected only 64 videos while failing to detect 236 fake videos. These results clearly show that the model is much better at recognizing real videos than identifying fake ones.

One possible reason for the lower performance is the domain difference between the FaceForensics++ dataset used for training and the Celeb-DF dataset used for testing. FaceForensics++ mainly contains manipulated videos with more visible compression artifacts and clearer synthetic patterns, while Celeb-DF was designed to produce more realistic and higher-quality deepfakes with fewer visual inconsistencies. In addition, Celeb-DF includes different lighting conditions, facial expressions, video qualities, and manipulation styles compared to FaceForensics++. As a result, the model may have learned patterns that are specific to FaceForensics++ instead of more general deepfake features, making it harder for the model to generalize well to Celeb-DF.

Another possible limitation comes from the temporal fusion method used in the model. The model uses mean pooling across frame features, meaning that all frames contribute equally to the final prediction. However, in deepfake videos, only a small number of frames may contain strong manipulation artifacts. Averaging all frames together can weaken these important signals and reduce fake detection performance.

The results also suggest that the classification threshold of 0.5 may not be optimal for Celeb-DF. Since the ROC-AUC score is significantly higher than the accuracy score, the model appears to assign somewhat higher probabilities to fake videos, but these probabilities may still remain below the threshold required for classification as fake. This indicates that threshold calibration or probability adjustment could improve recall and overall performance.

Overall, the evaluation demonstrates that the True Late Fusion model learned partially transferable deepfake features and achieved moderate cross-dataset discrimination ability. However, the model shows a strong bias toward predicting videos as real and struggles to detect many unseen fake samples. Future improvements may include better temporal fusion strategies, attention-based frame weighting, threshold optimization, and domain adaptation techniques to improve generalization on unseen datasets such as Celeb-DF.
